### Import Packages

In [ ]:
import os
import tifffile
import numpy as np
import nibabel as nib

In [ ]:
# Import basic packages for later use
import os
import shutil
from collections import OrderedDict

import json
import matplotlib.pyplot as plt
import nibabel as nib

import numpy as np
import torch

In [ ]:
# check whether GPU accelerated computing is available
assert torch.cuda.is_available() # if there is an error here, enable GPU in the Runtime

In [ ]:
torch.__version__

In [ ]:
# check if nnunet can be imported
import nnunetv2

In [ ]:
import importlib.metadata
version = importlib.metadata.version('nnunetv2')
print(f"nnunetv2 version: {version}")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

In [ ]:
def make_if_dont_exist(folder_path,overwrite=False):
    """
    creates a folder if it does not exists
    input: 
    folder_path : relative path of the folder which needs to be created
    over_write :(default: False) if True overwrite the existing folder 
    """
    if os.path.exists(folder_path):
        
        if not overwrite:
            print(f"{folder_path} exists.")
        else:
            print(f"{folder_path} overwritten")
            shutil.rmtree(folder_path)
            os.makedirs(folder_path)

    else:
      os.makedirs(folder_path)
      print(f"{folder_path} created!")

In [ ]:
import sys
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'deepbranchai_utils.py').exists() and (candidate / 'README.md').exists():
            return candidate
    raise RuntimeError('Could not find repository root')

REPO_DIR = find_repo_root()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from deepbranchai_utils import setup_environment

# Keep this as None to store data and nnU-Net folders under the repository.
# Set it to a large local drive folder to move assets to a data drive.
STORAGE_DIR = None

paths = setup_environment(REPO_DIR, storage_dir=STORAGE_DIR)
DATA_DIR = paths['data']
TMP_DIR = paths['tmp']
WEIGHTS_DIR = paths['weights']
NNUNET_RAW_DIR = paths['nnUNet_raw']
NNUNET_PREPROCESSED_DIR = paths['nnUNet_preprocessed']
NNUNET_RESULTS_DIR = paths['nnUNet_results']
STORAGE_ROOT = paths['storage']

root_dir = str(STORAGE_ROOT)


In [ ]:
mount_dir = root_dir

In [ ]:
# Maybe move path of preprocessed data directly on content - this may be signifcantely faster!
print("Current Working Directory {}".format(os.getcwd()))
path_dict = {
    "nnUNet_raw" : str(NNUNET_RAW_DIR), 
    "nnUNet_preprocessed" : str(NNUNET_PREPROCESSED_DIR), # 1 experiment: 1 epoch took 112s
    "nnUNet_results" : str(NNUNET_RESULTS_DIR),
    "RAW_DATA_PATH" : os.path.join(str(DATA_DIR), "RawData"), # This is used here only for convenience (not necessary for nnU-Net)!
}
# Write paths to environment variables

for env_var, path in path_dict.items():
  os.environ[env_var] = path 

# Check whether all environment variables are set correct!
for env_var, path in path_dict.items():
  if os.getenv(env_var) != path:
    print("Error:")
    print("Environment Variable {} is not set correctly!".format(env_var))
    print("Should be {}".format(path))
    print("Variable is {}".format(os.getenv(env_var)))
  make_if_dont_exist(path, overwrite=False)

print("If No Error Occured Continue Forward. =)")

In [ ]:
# Create Folderstructure for the new dataset!
dataset_name = 'Dataset4005_Mitochondria' #change here for different dataset name
nnunet_raw_data = os.path.join(os.getenv("nnUNet_raw"))
task_folder_name = os.path.join(nnunet_raw_data,dataset_name)
train_image_dir = os.path.join(task_folder_name,'imagesTr')
train_label_dir = os.path.join(task_folder_name,'labelsTr')
test_dir = os.path.join(task_folder_name,'imagesTs')

# Create Folder Structure for the SCGM dataset on the system
make_if_dont_exist(task_folder_name)
make_if_dont_exist(train_image_dir)
make_if_dont_exist(train_label_dir)
make_if_dont_exist(test_dir)

In [ ]:
training_data_name="Dataset4005_Mitochondria"
test_data_name="Dataset4005_Mitochondria"

In [ ]:
base_dir = DATA_DIR / 'TRAINING_SET_FINAL'

In [ ]:
os.chdir(base_dir)

## 3D Data Prep

In [ ]:
import os
import random
import numpy as np
import nibabel as nib
import tifffile
import shutil
import json

def get_valid_starts(dim_size, chunk_size, stride, min_shift_fraction=0.1):
    """
    Returns a list of valid start indices for a chunk of 'chunk_size' along
    a given dimension of length 'dim_size'. Increments by 'stride'.
    If the final chunk would go out of bounds, consider adding one last
    start so that the chunk exactly ends at 'dim_size' if the leftover space
    is >= min_shift_fraction * chunk_size.
    """
    if chunk_size > dim_size:
        return []  # cannot fit even one chunk

    starts = list(range(0, dim_size - chunk_size + 1, stride))
    last_start = dim_size - chunk_size
    if starts and (starts[-1] != last_start):
        # leftover space from starts[-1] to last_start
        shift_amount = last_start - starts[-1]
        threshold = min_shift_fraction * chunk_size
        if shift_amount >= threshold:
            starts.append(last_start)
    elif not starts:
        # No starts found by range; possibly chunk_size == dim_size
        if chunk_size == dim_size:
            starts = [0]

    return starts

def process_and_save_chunks(
    stack,
    chunks_dir,
    count,
    channel_name,
    stride=(32, 256, 256),
    chunk_size=(64, 512, 512),
    min_shift_fraction=0.1,
    test_mode=False
):
    """
    Extracts 3D chunks of size 'chunk_size' from 'stack', stepping by 'stride'.
    If leftover boundary space >= min_shift_fraction * chunk_size, a final
    chunk is added so the chunk extends exactly to the boundary.

    - If 'test_mode' is True => NO files are actually written.
    - Otherwise => Each chunk is saved as a .nii.gz file.

    Returns:
      - updated 'count'
      - list of chunk file names (or "[TEST_MODE] <name>" placeholders)
    """
    z, y, x = stack.shape
    print(f"\n[DEBUG] process_and_save_chunks() for '{channel_name}'")
    print(f"        stack.shape   = (z={z}, y={y}, x={x})")
    print(f"        chunk_size    = {chunk_size}, stride = {stride}, min_shift_fraction={min_shift_fraction}")

    # Compute valid starts along each dimension
    z_starts = get_valid_starts(z, chunk_size[0], stride[0], min_shift_fraction)
    y_starts = get_valid_starts(y, chunk_size[1], stride[1], min_shift_fraction)
    x_starts = get_valid_starts(x, chunk_size[2], stride[2], min_shift_fraction)

    if not z_starts or not y_starts or not x_starts:
        print(f"[DEBUG] Skipping {channel_name}: cannot fit chunk_size {chunk_size} in shape {(z, y, x)}")
        return count, []

    # Build a list of all valid (z, y, x) start positions
    all_starts = []
    for start_z in z_starts:
        for start_y in y_starts:
            for start_x in x_starts:
                all_starts.append((start_z, start_y, start_x))

    total_possible = len(all_starts)
    print(f"[DEBUG] Will extract up to {total_possible} chunks for '{channel_name}'.")

    chunk_filenames = []
    for i, (start_z, start_y, start_x) in enumerate(all_starts):
        end_z = start_z + chunk_size[0]
        end_y = start_y + chunk_size[1]
        end_x = start_x + chunk_size[2]

        chunk = stack[start_z:end_z, start_y:end_y, start_x:end_x]
        if chunk.shape != chunk_size:
            print(f"[DEBUG] Skipped chunk at (z={start_z},y={start_y},x={start_x}) -> shape mismatch {chunk.shape}")
            continue

        # Normalize or binarize
        if channel_name == 'raw':
            cmin, cmax = chunk.min(), chunk.max()
            if cmax > cmin:
                chunk = (chunk - cmin) / (cmax - cmin) * 255
            else:
                chunk = np.zeros_like(chunk)
            chunk = chunk.astype(np.uint8)
        else:  # channel_name == 'label'
            chunk = np.where(chunk > 0, 1, 0).astype(np.uint8)

        # Construct output name
        count_str = f"{count:05d}"
        if channel_name == "raw":
            chunk_name = os.path.join(chunks_dir, f"mitochondria_{count_str}_0000.nii.gz")
        else:
            chunk_name = os.path.join(chunks_dir, f"mitochondria_{count_str}.nii.gz")

        # Write chunk or mock write for test_mode
        if test_mode:
            # Just store the would-be filename
            print(f"[TEST_MODE] Would save chunk #{count} at (z={start_z}, y={start_y}, x={start_x}) to: {chunk_name}")
            chunk_filenames.append(f"[TEST_MODE] {os.path.basename(chunk_name)}")
        else:
            nib.save(nib.Nifti1Image(chunk, np.eye(4)), chunk_name)
            chunk_filenames.append(chunk_name)

            # Print progress every 10 chunks
            if (i + 1) % 10 == 0:
                print(f"    {i + 1}/{total_possible} chunks processed for {channel_name}")

        count += 1

    print(f"[DEBUG] Total chunks (for {channel_name}) after this call: {count}")
    return count, chunk_filenames

def main(directory, raw_chunks_dir, label_chunks_dir, test_mode=False):
    # Gather TIFF files
    files = [f for f in os.listdir(directory)
             if f.lower().endswith('.tiff') or f.lower().endswith('.tif')]
    random.shuffle(files)

    print("Files to be processed (in random order):")
    for f in files:
        print(f)

    proceed = input("\nDo you want to continue with the processing? (yes/no): ").strip().lower()
    if proceed != 'yes':
        print("Processing aborted by the user.")
        return

    # Determine starting count for raw
    existing_raw_files = [f for f in os.listdir(raw_chunks_dir) if f.endswith('.nii.gz')]
    if existing_raw_files:
        counts = []
        for fname in existing_raw_files:
            # Expect 'mitochondria_{count}_0000.nii.gz'
            parts = fname.split('_')
            try:
                count_val = int(parts[1])
                counts.append(count_val)
            except:
                pass
        count_raw = max(counts) + 1 if counts else 0
    else:
        count_raw = 0

    # Determine starting count for label
    existing_label_files = [f for f in os.listdir(label_chunks_dir) if f.endswith('.nii.gz')]
    if existing_label_files:
        counts = []
        for fname in existing_label_files:
            # Expect 'mitochondria_{count}.nii.gz'
            parts = fname.split('_')
            try:
                count_str_with_ext = parts[1]  # e.g. '00005.nii.gz'
                count_val = int(count_str_with_ext.split('.')[0])
                counts.append(count_val)
            except:
                pass
        count_labels = max(counts) + 1 if counts else 0
    else:
        count_labels = 0

    chunk_source_mapping_3dmito = {}

    # Process each file
    for file_path in files:
        print('\nPROCESSING:', file_path)
        full_path = os.path.join(directory, file_path)

        # Attempt to load
        try:
            stack = tifffile.imread(full_path)
            # If there's a dim == 2 or == 3, move that axis to last
            try:
                color_dim = np.where(np.array(stack.shape) == 2)[0][0]
                stack = np.moveaxis(stack, color_dim, -1)
                print('[DEBUG] Found dimension of size 2. Moved axis to end.')
            except:
                color_dim = np.where(np.array(stack.shape) == 3)[0][0]
                stack = np.moveaxis(stack, color_dim, -1)
                print('[DEBUG] Found dimension of size 3. Moved axis to end.')
            print(f"[DEBUG] Loaded '{file_path}' with shape={stack.shape}")

        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            continue

        # Extract raw (green channel)
        raw_data = stack[..., 1]
        labels = stack[..., 0]
        
        print(f"[DEBUG] raw_data.shape = {raw_data.shape}")
        print("[INFO] Using red channel for labels.")

        if labels.size == 0:
            print("[WARN] Label data is empty. Skipping file.")
            continue

        labels[labels > 1] = 1  # binarize >1 => 1
        print(f"[DEBUG] labels.shape = {labels.shape}")

        # RAW
        count_raw, raw_chunk_filenames = process_and_save_chunks(
            stack=raw_data,
            chunks_dir=raw_chunks_dir,
            count=count_raw,
            channel_name="raw",
            stride=(64, 176, 176),     # 50% overlap
            chunk_size=(128, 352, 352), # original chunk sizes
            min_shift_fraction=0.1,    # leftover must be >= 10%
            test_mode=test_mode
        )

        # LABEL
        count_labels, label_chunk_filenames = process_and_save_chunks(
            stack=labels,
            chunks_dir=label_chunks_dir,
            count=count_labels,
            channel_name="label",
            stride=(64, 176, 176),
            chunk_size=(128, 352, 352),
            min_shift_fraction=0.1,
            test_mode=test_mode
        )

        # Map chunk basenames => source file
        for chunk_filename in raw_chunk_filenames:
            base = os.path.basename(chunk_filename)
            chunk_source_mapping_3dmito[base] = file_path

        for chunk_filename in label_chunk_filenames:
            base = os.path.basename(chunk_filename)
            chunk_source_mapping_3dmito[base] = file_path

    # Write mapping if not test mode
    if not test_mode:
        mapping_filename = str(TMP_DIR / "chunk_source_mapping_3dmito.json")
        with open(mapping_filename, 'w') as f:
            json.dump(chunk_source_mapping_3dmito, f, indent=4)
        print(f"\nSaved chunk-source mapping to '{mapping_filename}'.")
    else:
        print("\n[TEST_MODE] No mapping file written.")

    print("\nProcessing complete!")

In [ ]:
import pandas as pd

# show all rows and columns
pd.set_option('display.max_rows',    None)   # or e.g. 1000
pd.set_option('display.max_columns', None)   # or a specific number
# optional—allow very wide tables without wrapping:
pd.set_option('display.width',        None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

In [ ]:
# %% [markdown]
# ## Scan for TIFF files, collect dimensions & compute summary statistics

# %%
from pathlib import Path
import pandas as pd
import tifffile

# directory to scan
base_dir = DATA_DIR / 'TRAINING_SET_FINAL'

records = []

for tif_path in base_dir.rglob('*'):
    if tif_path.suffix.lower() in ('.tif', '.tiff'):
        try:
            with tifffile.TiffFile(str(tif_path)) as tif:
                depth = len(tif.pages)
                first = tif.pages[0].asarray()
                height, width = first.shape[:2]
                channels = first.shape[2] if first.ndim == 3 else 1
        except Exception as e:
            print(f"Could not read {tif_path.name}: {e}")
            continue

        records.append({
            'filename': tif_path.name,
            'width':      width,
            'height':     height,
            'depth':      depth,
            'channels':   channels,
        })

# build a DataFrame
df = pd.DataFrame(records)

# display the full table of filenames and their metrics
print(f"Found {len(df)} TIFF files:\n")
display(df)

# compute and display summary statistics for numeric columns
print("\nSummary statistics for dimensions and channels:")
stats = df[['width', 'height', 'depth', 'channels']].describe().astype(int)
display(stats)

In [ ]:
directory = base_dir
raw_chunks_dir = str(NNUNET_RAW_DIR / 'Dataset4005_Mitochondria' / 'imagesTr')
label_chunks_dir = str(NNUNET_RAW_DIR / 'Dataset4005_Mitochondria' / 'labelsTr')

# Prompt user for test mode
test_input = input("Do you want to run in TEST MODE (no files will be written)? (yes/no): ").strip().lower()
test_mode = (test_input == "yes")

# Ensure chunk directories
for dir_path in [raw_chunks_dir, label_chunks_dir]:
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

# Optionally clear directories if not in test mode
if not test_mode:
    for dir_path in [raw_chunks_dir, label_chunks_dir]:
        files_in_dir = os.listdir(dir_path)
        if files_in_dir:
            print(f"The directory '{dir_path}' is not empty and contains {len(files_in_dir)} files.")
            proceed = input(f"Do you want to clear the directory '{dir_path}'? (yes/no): ").strip().lower()
            if proceed == 'yes':
                for filename in files_in_dir:
                    file_path = os.path.join(dir_path, filename)
                    try:
                        if os.path.isfile(file_path) or os.path.islink(file_path):
                            os.unlink(file_path)
                        elif os.path.isdir(file_path):
                            shutil.rmtree(file_path)
                    except Exception as e:
                        print(f'Failed to delete {file_path}. Reason: {e}')
                print(f"Directory '{dir_path}' has been cleared.")
            else:
                print(f"Directory '{dir_path}' will NOT be cleared.")
        else:
            print(f"The directory '{dir_path}' is empty.")

# Run the main code
main(directory, raw_chunks_dir, label_chunks_dir, test_mode=test_mode)

## Verification of Data

Before going any further, verify that the data is present and labels and data matches.

In [ ]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

def plot_segmentation_histogram(slices_dir):
    # List all NIfTI files in the directory
    files = [f for f in os.listdir(slices_dir) if f.endswith('.nii.gz') or f.endswith('.nii')]
    files.sort()
    
    proportions = []

    for file_name in files:
        file_path = os.path.join(slices_dir, file_name)
        
        try:
            # Load the NIfTI file
            nii = nib.load(file_path)
            data = nii.get_fdata()
            
            # Ensure data is binary (0 or 1)
            data = np.where(data > 0, 1, 0)
            
            # Compute the proportion of labeled voxels in this chunk
            total_voxels = data.size
            labeled_voxels = np.count_nonzero(data)
            proportion = labeled_voxels / total_voxels
            proportions.append(proportion)
        except Exception as e:
            print(f"Error loading {file_path}. Error: {e}")
            continue

    # Plot histogram
    plt.figure(figsize=(10, 6))
    plt.hist(proportions, bins=100, edgecolor='black')
    plt.xlabel('Proportion of Labeled Voxels in Slice', fontsize=16)
    plt.ylabel('Number of Slices', fontsize=16)
    plt.title('Histogram of Segmentation Proportions in Slices', fontsize=18)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_segmentation_histogram(train_label_dir)

In [ ]:
import os

# Define the folder path
folder_path = train_label_dir

print(folder_path)

# List all the .nii.gz files in the directory that start with 'mitochondria_' and end with '_0000.nii.gz'
nii_files = [f for f in os.listdir(folder_path) if f.startswith('mitochondria_') and f.endswith('_0000.nii.gz')]

# Loop through each .nii.gz file and rename
for nii_file in nii_files:
    # Construct the full path of the file
    full_path = os.path.join(folder_path, nii_file)
    
    # Remove the '_0000' from the file name and construct the new name
    new_file_name = nii_file.replace('_0000.nii.gz', '.nii.gz')
    new_full_path = os.path.join(folder_path, new_file_name)
    
    # Rename the file
    os.rename(full_path, new_full_path)

print("Renaming complete!")

In [ ]:
print(train_image_dir)
print(train_label_dir)

In [ ]:
train_files = os.listdir(train_image_dir)
label_files = os.listdir(train_label_dir)
print("train image files:",len(train_files))
print("train label files:",len(label_files))
print("Intersections:",len(set(train_files).intersection(set(label_files))))

In [ ]:
#renaming to add the modality for SCGM there is only one modality 
#images should be added with 0000
#can be skipped if modality is already mentioned
#re-write for multiple modalities

def check_modality(filename):
    """
    check for the existence of modality
    return False if modality is not found else True
    """
    end = filename.find('.nii.gz')
    modality = filename[end-4:end]
    for mod in modality: 
        if not(ord(mod)>=48 and ord(mod)<=57): #if not in 0 to 9 digits
            return False
    return True

def rename_for_single_modality(directory):
    
    for file in os.listdir(directory):
        
        if check_modality(file)==False:
            new_name = file[:file.find('.nii.gz')]+"_0000.nii.gz"
            os.rename(os.path.join(directory,file),os.path.join(directory,new_name))
            print(f"Renamed to {new_name}")

rename_for_single_modality(train_image_dir)

In [ ]:
task_folder_name

## Preprocessing

In [ ]:
overwrite_json_file = True #make it True if you want to overwrite the dataset.json file in Dataset_folder
json_file_exist = False

if os.path.exists(os.path.join(task_folder_name,'dataset.json')):
    print('dataset.json already exist!')
    json_file_exist = True

if json_file_exist==False or overwrite_json_file:

    json_dict = OrderedDict()
    json_dict['name'] = dataset_name
    json_dict['description'] = "FIBSEM"
    json_dict['tensorImageSize'] = "3D"
    json_dict['reference'] = "N/A"
    json_dict['licence'] = "N/A"
    json_dict['release'] = "0.0"

    #you may mention more than one modality
    json_dict['channel_names'] = {
        "0": "SEM"
    }

    # set expected file ending
    json_dict["file_ending"] = ".nii.gz"

    #label names should be mentioned for all the labels in the dataset
    json_dict['labels'] = {
        "background": 0,
        "mitochondria": 1,
    }
    
    train_ids = os.listdir(train_label_dir)
    test_ids = os.listdir(test_dir)
    json_dict['numTraining'] = len(train_ids)
    json_dict['numTest'] = len(test_ids)
    
    with open(os.path.join(task_folder_name,"dataset.json"), 'w') as f:
        json.dump(json_dict, f, indent=4, sort_keys=True)

    if os.path.exists(os.path.join(task_folder_name,'dataset.json')):
        if json_file_exist==False:
            print('dataset.json created!')
        else: 
            print('dataset.json overwritten!')

In [ ]:
main_dir = str(STORAGE_ROOT)

In [ ]:
os.environ['nnUNet_raw_data_base'] = str(NNUNET_RAW_DIR)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_DIR)
os.environ['RESULTS_FOLDER'] = str(NNUNET_RESULTS_DIR)

In [ ]:
!nnUNetv2_plan_and_preprocess -d 4005 -c 3d_fullres --verify_dataset_integrity

In [ ]:
import json

# Assuming 'nnUNetPlans.json' is in the same directory as your script
file_name = os.path.join(os.environ['nnUNet_preprocessed'], dataset_name, 'nnUNetPlans.json')

# Load the JSON data from the file
with open(file_name, 'r') as file:
    data = json.load(file)

# Modify the 'patch_size' under 'configurations' -> '3d'
data['configurations']['3d_fullres']['patch_size'] = [352, 352, 128]
data['configurations']['3d_fullres']['batch_size'] = 2
#data['configurations']['3d_fullres']['median_image_size_in_voxels'] = [352, 352, 128]


# Save the modified data back to the file
with open(file_name, 'w') as file:
    json.dump(data, file, indent=4)

In [ ]:
import json
import os
import random
from collections import Counter

# ——— CONFIG ———
json_path = str(TMP_DIR / "chunk_source_mapping_3dmito.json")
dataset_name = 'Dataset4005_Mitochondria'  # Adjust this to your dataset name

# Define subject groups based on the file groupings
SUBJECT_GROUPS = {
    'subject_073': [
        'b3GT073_160715-a-327-454_combined.tiff',  # lowercase 'b'
        'b3GT073_bl1_161015-a_-211-338_combined.tiff'  # 'bl1' not 'bt1'
    ],
    'subject_029b1': [
        'B3Gt029B1_a2_try3_731-858_combined.tiff',  # lowercase 't'
        'B3Gt029B1_a3b_try4_20-147_combined.tiff'   # lowercase 't'
    ],
    'subject_035v3': [
        'B3Gt035V3_area3_try2_converted-1-128_combined.tiff',  # lowercase 't'
        'B3Gt035V3_capnxtdoor4-70-197_combined.tiff'  # lowercase 't'
    ],
    'subject_061': [
        'B3Gt061_1_A5v2_78-205_combined.tiff',  # lowercase 't'
        'B3Gt061PbV3_A7_converted_381-508_combined.tiff'  # lowercase 't'
    ],
    'subject_089v3': [
        'B3Gt089v3_a1_3D-588-715_combined.tiff',  # lowercase 't'
        'B3Gt089v3_a3_try3-1218-1345_combined.tiff'  # lowercase 't'
    ],
    'subject_108': [
        'B3Gt108V2_B1_230607B_T4-1-128_combined.tiff',  # lowercase 't'
        'B3GT108V3_230703_5-132_combined.tiff'
    ],
    'subject_084': [
        'B3GT084_Bb2a_170415-a_-13-140_combined.tiff',
        'B3GT084_BB1_170722-a_-1-128_combined.tiff'
    ],
    'subject_101': [
        'B3GT101_220825-1-128_combined.tiff',
        'B3GT101_220920_converted_-1-128_combined.tiff'
    ],
    'subject_109': [
        'B3GT109_reda_170818-a_-30-157_combined.tiff',
        'B3GT109_redb_170816-a-1-128_combined.tiff'
    ],
    'B3TP149': [
        'B3TP149PB_3_7b_3D_-1-128_combined.tiff',
        'B3TP1491_1a_try2_-181-308_combined.tiff'
    ]
}

# ——— Step 1: Read the JSON file ———
if not os.path.exists(json_path):
    raise FileNotFoundError(f"JSON file not found: {json_path}")

with open(json_path, 'r') as f:
    data = json.load(f)

# ——— Step 2: Count unique volume names ———
values = list(data.values())
unique_values = set(values)
print(f'Number of unique values (volumes): {len(unique_values)}\n')

# ——— Step 3: Display each value and its counts ———
value_counts = Counter(values)
for vol, cnt in value_counts.items():
    print(f'{vol}: {cnt}')
print()

# ——— Step 4: Map volume_name → list of chunk keys ———
value_to_keys = {}
for chunk_key, vol in data.items():
    value_to_keys.setdefault(vol, []).append(chunk_key)

# ——— Step 5: Map volumes to subjects ———
volume_to_subject = {}
subject_to_volumes = {}

# First, create a mapping from volumes to subjects
for subject, volume_list in SUBJECT_GROUPS.items():
    subject_to_volumes[subject] = []
    for volume in volume_list:
        # Check if this volume exists in our data
        if volume in unique_values:
            volume_to_subject[volume] = subject
            subject_to_volumes[subject].append(volume)
        else:
            print(f"Warning: Volume {volume} not found in data")

# Remove subjects with no volumes
subject_to_volumes = {s: v for s, v in subject_to_volumes.items() if v}

print(f"\nSubject mapping created:")
for subject, volumes in subject_to_volumes.items():
    print(f"  {subject}: {len(volumes)} volumes")
print()

# ——— Step 6: Create subject-wise folds ———
all_subjects = list(subject_to_volumes.keys())
random.shuffle(all_subjects)

num_folds = 5
n = len(all_subjects)
fold_size = n // num_folds
rem = n % num_folds

subject_folds = []
start = 0
for i in range(num_folds):
    size = fold_size + (1 if i < rem else 0)
    subject_folds.append(all_subjects[start:start+size])
    start += size

print(f"Subject distribution across folds:")
for i, subjects in enumerate(subject_folds):
    print(f"  Fold {i+1}: {subjects}")
print()

# ——— Step 7: Create splits and top-level summary ———
splits = []
toplevel = []

def clean_key(k):
    k = k.replace('.nii.gz', '')
    if k.endswith('_0000'):
        k = k[:-5]
    return k

for i in range(num_folds):
    val_subjects = subject_folds[i]
    train_subjects = [s for j, f in enumerate(subject_folds) if j != i for s in f]
    
    # Get volumes for validation and training subjects
    val_vols = [v for s in val_subjects for v in subject_to_volumes[s]]
    train_vols = [v for s in train_subjects for v in subject_to_volumes[s]]
    
    # Gather raw chunk keys
    val_raw = [k for v in val_vols for k in value_to_keys.get(v, [])]
    train_raw = [k for v in train_vols for k in value_to_keys.get(v, [])]
    
    # Clean & dedupe
    val_keys = sorted({clean_key(k) for k in val_raw})
    train_keys = sorted({clean_key(k) for k in train_raw})
    
    # Print fold info
    print(f"Fold {i+1}:")
    print(f"  Validation Subjects ({len(val_subjects)}): {val_subjects}")
    print(f"  Validation Volumes ({len(val_vols)}): {val_vols}")
    print(f"  Total validation keys: {len(val_keys)}")
    print(f"  Training Subjects ({len(train_subjects)}): {train_subjects}")
    print(f"  Training Volumes ({len(train_vols)}): {train_vols}")
    print(f"  Total training keys: {len(train_keys)}\n")
    
    # Accumulate JSON entries
    toplevel.append({
        "fold": i+1,
        "validation_subjects": val_subjects,
        "validation_volumes": val_vols,
        "total_validation_keys": len(val_keys),
        "training_subjects": train_subjects,
        "training_volumes": train_vols,
        "total_training_keys": len(train_keys)
    })
    
    splits.append({
        "train": train_keys,
        "val": val_keys
    })

# ——— Step 8: Write both JSON files ———
base_dir = os.path.join(os.environ.get('nnUNet_preprocessed', '.'), dataset_name)
os.makedirs(base_dir, exist_ok=True)

manual_path = os.path.join(base_dir, 'splits_final.json')
toplevel_path = os.path.join(base_dir, 'splits_toplevel.json')

with open(manual_path, 'w') as f:
    json.dump(splits, f, indent=4)

with open(toplevel_path, 'w') as f:
    json.dump(toplevel, f, indent=4)

print(f"Splits have been saved to:\n • {manual_path}\n • {toplevel_path}")

## Training nnU-Net

Generic Training Commands:

```nnUNetv2_train Dataset_NAME_OR_ID CONFIGURATION FOLD -tr TRAINER_CLASS_NAME (additional options)```

For 2D:  ```nnUNetv2_train DATASET_NAME_OR_ID 2d FOLD```

For 3D Full resolution: ```nnUNetv2_train DATASET_NAME_OR_ID 3d_fullres FOLD```

For Cascaded 3D:

First Run lowres: ```nnUNetv2_train DATASET_NAME_OR_ID 3d_lowres FOLD```

Then Run fullres: ```nnUNetv2_train DATASET_NAME_OR_ID 3d_cascade_fullres FOLD```

In [ ]:
!nnUNetv2_train --help

In [ ]:
!nnUNetv2_train 4005 3d_fullres 0 -tr nnUNetTrainer_100epochs --c

In [ ]:
!nnUNetv2_train 4005 3d_fullres 1 -tr nnUNetTrainer_100epochs --c

In [ ]:
!nnUNetv2_train 4005 3d_fullres 2 -tr nnUNetTrainer_100epochs --c

In [ ]:
!nnUNetv2_train 4005 3d_fullres 3 -tr nnUNetTrainer_100epochs --c

In [ ]:
!nnUNetv2_train 4005 3d_fullres 4 -tr nnUNetTrainer_100epochs --c